# Steering Effectiveness Analysis
Loads all episode logs from `logs/`, parses step-level and episode-level data, and produces breakdowns by layer, game, steering method, and agent.

In [ ]:
import json, re, os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sns.set_theme(style='whitegrid', palette='tab10')
plt.rcParams['figure.dpi'] = 120

LOGS_DIR = Path('../logs')

## 1. Load data

In [ ]:
def parse_run_id(run_id):
    game = 'gbs' if run_id.startswith('gbs_') else 'beauty_contest'
    if 'activation' in run_id:  steering = 'activation'
    elif 'prompt' in run_id:    steering = 'prompt'
    else:                        steering = 'noop'
    scope = 'one' if '_one' in run_id else 'all'
    directional = 'directional' in run_id
    m = re.search(r'_l(\d+)$', run_id)
    layer = int(m.group(1)) if m else None
    return dict(game=game, steering=steering, scope=scope,
                directional=directional, layer=layer)


step_rows, summary_rows = [], []

for run_dir in sorted(LOGS_DIR.iterdir()):
    if not run_dir.is_dir():
        continue
    meta = parse_run_id(run_dir.name)

    for summary_path in sorted(run_dir.glob('episode_*.summary.json')):
        ep = int(re.search(r'episode_(\d+)', summary_path.name).group(1))
        try:
            summary = json.loads(summary_path.read_text())
        except Exception:
            continue
        summary_rows.append({**meta, 'run_id': run_dir.name, 'episode': ep, **{
            'final_rewards': summary.get('final_rewards', {}),
            'gbs_converged': summary.get('gbs_converged'),
            'gbs_converged_round': summary.get('gbs_converged_round'),
        }})

        jsonl_path = summary_path.with_suffix('').with_suffix('.jsonl')
        if not jsonl_path.exists():
            continue
        for line in jsonl_path.read_text().splitlines():
            try:
                rec = json.loads(line)
            except Exception:
                continue
            info = rec.get('info', {})
            step_rows.append({
                **meta,
                'run_id':      run_dir.name,
                'episode':     ep,
                'turn':        rec.get('turn'),
                'agent_id':    rec.get('agent_id'),
                'action':      rec.get('parsed_action'),
                'reward':      rec.get('reward'),
                'retries':     rec.get('parse_retries', 0),
                'is_steered':  rec.get('steering_spec_id') not in (None, 'noop'),
                # beauty contest
                'bc_mean':     info.get('mean'),
                'bc_target':   info.get('target'),
                'bc_winners':  info.get('winners'),
                # gbs
                'gbs_group_sum': info.get('group_sum'),
                'gbs_error':     info.get('error'),
                'gbs_direction': info.get('direction'),
            })

steps = pd.DataFrame(step_rows)
summaries = pd.DataFrame(summary_rows)

# cast action to numeric where possible
steps['action_num'] = pd.to_numeric(steps['action'], errors='coerce')

print(f'Steps:     {len(steps):,}')
print(f'Summaries: {len(summaries):,}')
print(f'Run ids:   {steps["run_id"].nunique()}')
steps.head(3)

In [ ]:
# Convenience subsets
bc    = steps[steps.game == 'beauty_contest'].copy()
gbs   = steps[steps.game == 'gbs'].copy()
bc_s  = summaries[summaries.game == 'beauty_contest'].copy()
gbs_s = summaries[summaries.game == 'gbs'].copy()

# Mark whether each agent was the steered one (player_0 in 'one' configs)
for df in (bc, gbs):
    df['is_player0'] = df['agent_id'] == 'player_0'
    df['steered_agent'] = ((df['scope'] == 'one') & df['is_player0']) | (df['scope'] == 'all')

layers = sorted(steps['layer'].dropna().unique().astype(int))
print('Layers with data:', layers)

## 2. Beauty contest
Lower guesses → closer to Nash equilibrium (0). Winning = closest to 2/3 of group mean.

In [ ]:
# ── Mean guess by layer and steering method ───────────────────────────────────
bc_act = bc[bc.steering == 'activation']
bc_agg = (bc_act.groupby(['layer', 'scope'])['action_num']
          .mean().reset_index(name='mean_guess'))

# baselines (no layer)
bc_base = bc[bc.steering != 'activation']
bc_base_agg = (bc_base.groupby('steering')['action_num']
               .mean().reset_index(name='mean_guess'))

fig, ax = plt.subplots(figsize=(11, 4))
for scope, grp in bc_agg.groupby('scope'):
    grp = grp.sort_values('layer')
    ax.plot(grp['layer'], grp['mean_guess'], marker='o', label=f'activation ({scope})')

for _, row in bc_base_agg.iterrows():
    ax.axhline(row['mean_guess'], linestyle='--', label=row['steering'])

ax.set_xlabel('Layer')
ax.set_ylabel('Mean guess')
ax.set_title('Beauty Contest — mean guess by layer (lower = more ToM-rational)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Steered (player_0) vs unsteered players — guess distributions ─────────────
bc_one = bc[(bc.scope == 'one') & (bc.steering == 'activation')].copy()
bc_one['role'] = bc_one['is_player0'].map({True: 'player_0 (steered)', False: 'others'})

if not bc_one.empty:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    # overall distribution
    sns.boxplot(data=bc_one, x='role', y='action_num', ax=axes[0],
                order=['player_0 (steered)', 'others'])
    axes[0].set_title('Guess distribution: steered vs unsteered (all layers)')
    axes[0].set_xlabel('')
    axes[0].set_ylabel('Guess')

    # by layer
    layer_role = (bc_one.groupby(['layer', 'role'])['action_num']
                  .mean().reset_index(name='mean_guess'))
    for role, grp in layer_role.groupby('role'):
        grp = grp.sort_values('layer')
        axes[1].plot(grp['layer'], grp['mean_guess'], marker='o', label=role)
    axes[1].set_xlabel('Layer')
    axes[1].set_ylabel('Mean guess')
    axes[1].set_title('Mean guess by layer: steered vs unsteered')
    axes[1].legend()

    plt.tight_layout()
    plt.show()
else:
    print('No activation/one beauty_contest data yet.')

In [ ]:
# ── Win rate of player_0 by layer (one-agent steering configs) ────────────────
def won(row):
    winners = row.get('bc_winners') or []
    return int('player_0' in winners) if winners else np.nan

bc_one_steps = steps[(steps.game == 'beauty_contest') &
                     (steps.scope == 'one') &
                     (steps.steering == 'activation') &
                     (steps.agent_id == 'player_0')].copy()
bc_one_steps['won'] = bc_one_steps.apply(lambda r: won(r), axis=1)

if not bc_one_steps.empty:
    winrate = (bc_one_steps.dropna(subset=['won'])
               .groupby('layer')['won'].mean() * 100)
    noop_winrate = (steps[(steps.game == 'beauty_contest') &
                          (steps.steering == 'noop') &
                          (steps.agent_id == 'player_0')]
                   .apply(lambda r: won(r), axis=1).dropna().mean() * 100)

    fig, ax = plt.subplots(figsize=(11, 4))
    winrate.plot(marker='o', ax=ax, label='activation (one)')
    ax.axhline(noop_winrate, linestyle='--', color='grey', label=f'noop baseline ({noop_winrate:.1f}%)')
    ax.axhline(25, linestyle=':', color='black', alpha=0.4, label='chance (25%)')
    ax.set_xlabel('Layer')
    ax.set_ylabel('player_0 win rate (%)')
    ax.set_title('Beauty Contest — player_0 win rate by layer')
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print('No activation/one beauty_contest data yet.')

In [ ]:
# ── Guess error (|guess - target|) per layer ──────────────────────────────────
bc['guess_error'] = (bc['action_num'] - bc['bc_target']).abs()
bc_one2 = bc[(bc.scope == 'one') & (bc.steering == 'activation')].copy()
bc_one2['role'] = bc_one2['is_player0'].map({True: 'player_0 (steered)', False: 'others'})

if not bc_one2.empty:
    err_agg = (bc_one2.groupby(['layer', 'role'])['guess_error']
               .mean().reset_index(name='mean_error'))
    fig, ax = plt.subplots(figsize=(11, 4))
    for role, grp in err_agg.groupby('role'):
        ax.plot(grp['layer'], grp['mean_error'], marker='o', label=role)
    ax.set_xlabel('Layer')
    ax.set_ylabel('Mean |guess - target|')
    ax.set_title('Beauty Contest — guess error by layer (lower = better coordination)')
    ax.legend()
    plt.tight_layout()
    plt.show()

## 3. GBS — Goldstone Group Sum
Group wins when contributions sum to the hidden target. Primary metric: convergence rate and rounds to converge.

In [ ]:
# ── Convergence rate by layer and steering ────────────────────────────────────
gbs_s['converged'] = gbs_s['gbs_converged'].fillna(False).astype(bool)

gbs_conv = (gbs_s[gbs_s.steering == 'activation']
            .groupby(['layer', 'scope'])['converged']
            .mean().reset_index(name='conv_rate'))
gbs_base_conv = (gbs_s[gbs_s.steering != 'activation']
                 .groupby(['steering', 'directional'])['converged']
                 .mean().reset_index(name='conv_rate'))

fig, ax = plt.subplots(figsize=(11, 4))
for scope, grp in gbs_conv.groupby('scope'):
    grp = grp.sort_values('layer')
    ax.plot(grp['layer'], grp['conv_rate'] * 100, marker='o', label=f'activation ({scope})')

for _, row in gbs_base_conv.iterrows():
    label = row['steering'] + (' (dir)' if row['directional'] else '')
    ax.axhline(row['conv_rate'] * 100, linestyle='--', label=label)

ax.set_xlabel('Layer')
ax.set_ylabel('Convergence rate (%)')
ax.set_title('GBS — convergence rate by layer')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Rounds to converge (episodes that did converge) ───────────────────────────
gbs_conv_ep = gbs_s[gbs_s['converged'] & gbs_s['layer'].notna()]

if not gbs_conv_ep.empty:
    conv_rd = (gbs_conv_ep.groupby(['layer', 'scope'])['gbs_converged_round']
               .mean().reset_index(name='mean_round'))

    fig, ax = plt.subplots(figsize=(11, 4))
    for scope, grp in conv_rd.groupby('scope'):
        grp = grp.sort_values('layer')
        ax.plot(grp['layer'], grp['mean_round'], marker='o', label=f'activation ({scope})')

    for steering in ('noop', 'prompt'):
        base = gbs_s[(gbs_s.steering == steering) & gbs_s['converged']]
        if not base.empty:
            ax.axhline(base['gbs_converged_round'].mean(), linestyle='--', label=steering)

    ax.set_xlabel('Layer')
    ax.set_ylabel('Mean round of convergence')
    ax.set_title('GBS — rounds to converge by layer (lower = faster, converged episodes only)')
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print('No convergence data yet.')

In [ ]:
# ── GBS: player_0 vs others contribution ─────────────────────────────────────
gbs['contribution'] = gbs['action_num']
gbs_one = gbs[(gbs.scope == 'one') & (gbs.steering == 'activation')].copy()
gbs_one['role'] = gbs_one['is_player0'].map({True: 'player_0 (steered)', False: 'others'})

if not gbs_one.empty:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    # overall distribution
    sns.boxplot(data=gbs_one, x='role', y='contribution', ax=axes[0],
                order=['player_0 (steered)', 'others'])
    axes[0].set_title('GBS contribution: steered vs unsteered (all layers)')
    axes[0].set_xlabel('')

    # by layer
    cont_agg = (gbs_one.groupby(['layer', 'role'])['contribution']
                .mean().reset_index(name='mean_contribution'))
    for role, grp in cont_agg.groupby('role'):
        grp = grp.sort_values('layer')
        axes[1].plot(grp['layer'], grp['mean_contribution'], marker='o', label=role)
    axes[1].set_xlabel('Layer')
    axes[1].set_ylabel('Mean contribution')
    axes[1].set_title('GBS — mean contribution by layer: steered vs unsteered')
    axes[1].legend()

    plt.tight_layout()
    plt.show()
else:
    print('No activation/one GBS data yet.')

In [ ]:
# ── GBS abs error trajectory over rounds ─────────────────────────────────────
gbs['abs_error'] = gbs['gbs_error'].abs()

err_traj = (gbs.dropna(subset=['abs_error'])
            .groupby(['turn', 'steering'])['abs_error']
            .mean().reset_index())

fig, ax = plt.subplots(figsize=(10, 4))
for s, grp in err_traj.groupby('steering'):
    ax.plot(grp['turn'], grp['abs_error'], marker='o', label=s)
ax.set_xlabel('Turn (round × num_players)')
ax.set_ylabel('Mean |error|')
ax.set_title('GBS — absolute error over turns by steering method')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Steering lift — steered player vs unsteered
For 'one' configs: how much better does player_0 do compared to unsteered players in the same episodes?

In [ ]:
# Per episode: compute mean metric for player_0 vs mean of others
def compute_lift(df, game, metric_col, agg='mean'):
    """Returns a DataFrame with columns [layer, lift] where
    lift = metric(player_0) - metric(others) per episode, averaged over episodes."""
    sub = df[(df.game == game) & (df.scope == 'one') &
             (df.steering == 'activation') & df[metric_col].notna()].copy()
    if sub.empty:
        return pd.DataFrame(columns=['layer', 'lift'])

    ep_grp = sub.groupby(['run_id', 'episode', 'layer', 'is_player0'])[metric_col]
    ep_agg = ep_grp.mean().reset_index(name='val')
    p0  = ep_agg[ep_agg.is_player0].rename(columns={'val': 'p0_val'})
    oth = ep_agg[~ep_agg.is_player0].groupby(
        ['run_id', 'episode', 'layer'])['val'].mean().reset_index(name='oth_val')
    merged = p0.merge(oth, on=['run_id', 'episode', 'layer'])
    merged['lift'] = merged['p0_val'] - merged['oth_val']
    return merged.groupby('layer')['lift'].mean().reset_index()


bc_lift  = compute_lift(steps, 'beauty_contest', 'action_num')   # lower guess = more ToM
gbs_lift = compute_lift(steps, 'gbs', 'action_num')              # different contribution

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

if not bc_lift.empty:
    axes[0].bar(bc_lift['layer'], bc_lift['lift'], color='steelblue')
    axes[0].axhline(0, color='black', linewidth=0.8)
    axes[0].set_xlabel('Layer')
    axes[0].set_ylabel('player_0 guess − others mean guess')
    axes[0].set_title('BC steering lift (negative = player_0 guesses lower)')
else:
    axes[0].text(0.5, 0.5, 'No data', ha='center', va='center',
                 transform=axes[0].transAxes)

if not gbs_lift.empty:
    axes[1].bar(gbs_lift['layer'], gbs_lift['lift'], color='coral')
    axes[1].axhline(0, color='black', linewidth=0.8)
    axes[1].set_xlabel('Layer')
    axes[1].set_ylabel('player_0 contribution − others mean')
    axes[1].set_title('GBS steering lift (player_0 contribution deviation)')
else:
    axes[1].text(0.5, 0.5, 'No data', ha='center', va='center',
                 transform=axes[1].transAxes)

plt.tight_layout()
plt.show()

In [ ]:
# ── Win-rate lift for beauty contest ─────────────────────────────────────────
def bc_win_lift():
    sub = steps[(steps.game == 'beauty_contest') &
                (steps.scope == 'one') &
                (steps.steering == 'activation') &
                steps['bc_winners'].notna()].copy()
    if sub.empty:
        return pd.DataFrame()
    sub['won'] = sub.apply(lambda r: int(r['agent_id'] in (r['bc_winners'] or [])), axis=1)
    wr = sub.groupby(['layer', 'is_player0'])['won'].mean().reset_index(name='win_rate')
    p0  = wr[wr.is_player0].rename(columns={'win_rate': 'p0_wr'})[['layer', 'p0_wr']]
    oth = wr[~wr.is_player0].rename(columns={'win_rate': 'oth_wr'})[['layer', 'oth_wr']]
    merged = p0.merge(oth, on='layer')
    merged['lift_pp'] = (merged['p0_wr'] - merged['oth_wr']) * 100
    return merged

wl = bc_win_lift()
if not wl.empty:
    fig, ax = plt.subplots(figsize=(11, 4))
    ax.bar(wl['layer'], wl['lift_pp'], color='steelblue')
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_xlabel('Layer')
    ax.set_ylabel('Win-rate lift (pp)')
    ax.set_title('BC — player_0 win-rate lift over others by layer (pp = percentage points)')
    plt.tight_layout()
    plt.show()
    display(wl.sort_values('lift_pp', ascending=False).reset_index(drop=True))
else:
    print('No activation/one BC data yet.')

## 5. Summary table

In [ ]:
def summary_table(game):
    s = summaries[summaries.game == game].copy()
    s['n'] = 1
    rows = []
    for (steering, scope, layer), grp in s.groupby(['steering', 'scope', 'layer'],
                                                    dropna=False):
        row = dict(steering=steering, scope=scope,
                   layer=int(layer) if layer == layer else '-',
                   n_episodes=len(grp))
        if game == 'gbs':
            conv = grp['gbs_converged'].fillna(False)
            row['conv_rate_%']    = f"{conv.mean()*100:.1f}"
            rds = grp.loc[conv, 'gbs_converged_round']
            row['mean_conv_round'] = f"{rds.mean():.1f}" if len(rds) else '-'
        rows.append(row)
    return pd.DataFrame(rows).sort_values(
        ['steering', 'scope', 'layer'], na_position='last')

print('=== BEAUTY CONTEST ===')
display(summary_table('beauty_contest'))
print('\n=== GBS ===')
display(summary_table('gbs'))